In [2]:
from game import Game
import time
import os
import re

## Load maps

In [3]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [4]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [5]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [6]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [7]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [8]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [9]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [10]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

### State Implementation:

In [11]:
from enum import Enum
from dataclasses import dataclass
from typing import List, Tuple

@dataclass(frozen=True)
class State:
    player_pos: Tuple[int, int]
    boxes_pos: Tuple[Tuple[int, int], ...]
    solution_path: str = ""
    
    def get_boxes_pos(self) -> Tuple[Tuple[int, int]]:
        return self.boxes_pos

    def get_player_pos(self) -> Tuple[int, int]:
        return self.player_pos
    
    def get_solution_path(self) -> str:
        return self.solution_path
    
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, State):
            return False
        return self.player_pos == other.player_pos and self.boxes_pos == other.boxes_pos

    def __hash__(self) -> int:
        return hash((self.player_pos, self.boxes_pos))

class Direction(Enum):
    UP = 'U'
    DOWN = 'D'
    LEFT = 'L'
    RIGHT = 'R'

def next_states(game: Game, cur_state: State) -> list[State]:
    next_States = []
    game.set_player_position(cur_state.get_player_pos())
    game.set_box_positions(list(cur_state.get_boxes_pos()))

    for direction in Direction:
        if game.apply_move(direction.value):
            next_States.append(
                State(player_pos=game.get_player_position(),
                boxes_pos=tuple(tuple(box) for box in game.get_box_locations()), 
                solution_path=cur_state.solution_path + direction.value)
            )
            
            game.set_player_position(cur_state.get_player_pos())
            game.set_box_positions(list(cur_state.get_boxes_pos()))    

    return next_States


### BFS

In [12]:
from collections import deque 

def solver_bfs(game_map):
    game = Game(game_map)

    goal_positions = tuple(tuple(goal) for goal in game.get_goal_locations())
    def is_goal_state(boxes, goals):
        for i in range(len(goals)):
            if boxes[i] != goals[i]:
                return False
        return True

    queue = deque()
    visited_states = set() 

    initial_state = State(player_pos=game.get_player_position(), boxes_pos=tuple(tuple(box) for box in game.get_box_locations()), solution_path="")
    if game.is_game_won():
        return "", 0
    queue.append(initial_state)
    visited_states.add(initial_state)

    while queue:
        cur_state = queue.popleft()

        for next_state in next_states(game, cur_state):
            if next_state not in visited_states:
                if is_goal_state(next_state.get_boxes_pos(), goal_positions):
                    return next_state.get_solution_path(), len(visited_states)
      
                queue.append(next_state)
                visited_states.add(next_state)

    return "", len(visited_states)

### DFS

In [13]:
# TODO: Must return moves, number of visited states
def solver_dfs(game_map):
    game = Game(game_map)
    return None, 0

### IDS

In [14]:
# TODO: Must return moves, number of visited states
def solver_ids(game_map):
    game = Game(game_map)
    return None, 0

### A*

In [15]:
# TODO
def heuristic(game_map):
    return 0

# TODO: Must return moves, number of visited states
def solver_astar(game_map, heuristic_func=heuristic, weight=1):
    game = Game(game_map)
    return None, 0

## Solve

In [16]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
    "A*": solver_astar
}

In [17]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [21]:
solve(7, "BFS")

BFS took 8.07 seconds on map 7 and visited 644753 states.
34 moves were used: RURRDDDDLDRUUUULLLRDRDRDDLLDLLUUDR


In [19]:
def solve_all():
    for map in range(min(map_file_indices), max(map_file_indices) + 1):
        for method in SOLVERS.keys():
            solve(map, method)
            

In [20]:
solve_all() # Solve all maps using all methods

BFS took 0.0 seconds on map 1 and visited 43 states.
7 moves were used: UDDULRR
DFS took 0.0 seconds on map 1 and visited 0 states.
No Solution Found!
IDS took 0.0 seconds on map 1 and visited 0 states.
No Solution Found!
A* took 0.0 seconds on map 1 and visited 0 states.
No Solution Found!
BFS took 0.0 seconds on map 2 and visited 20 states.
6 moves were used: LUDDUL
DFS took 0.0 seconds on map 2 and visited 0 states.
No Solution Found!
IDS took 0.0 seconds on map 2 and visited 0 states.
No Solution Found!
A* took 0.0 seconds on map 2 and visited 0 states.
No Solution Found!
BFS took 0.0 seconds on map 3 and visited 121 states.
13 moves were used: ULDDUUUURDDDD
DFS took 0.0 seconds on map 3 and visited 0 states.
No Solution Found!
IDS took 0.0 seconds on map 3 and visited 0 states.
No Solution Found!
A* took 0.0 seconds on map 3 and visited 0 states.
No Solution Found!
BFS took 0.0 seconds on map 4 and visited 2 states.
0 moves were used: 
DFS took 0.0 seconds on map 4 and visited 0 s

KeyboardInterrupt: 